In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import importlib
import subprocess
import sys

def ensure_package(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for package_name in ["torch", "torchvision", "matplotlib", "numpy"]:
    ensure_package(package_name)

print("Environment ready.")

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

try:
    from torchvision.models import ResNet18_Weights, resnet18
except ImportError:
    from torchvision.models import resnet18
    ResNet18_Weights = None

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"ResNet18_Weights available: {ResNet18_Weights is not None}")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Two-Stream ConvNet 代码实战：学习实现 vs 工程实现

基于论文 *Two-Stream Convolutional Networks for Action Recognition in Videos*（Simonyan & Zisserman, NeurIPS 2014），
用一个轻量 **video action recognition** toy 任务演示 Two-Stream 的核心思想。

本 Notebook 包含两条并行路径，使用**同一任务**与**同一数据**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解空间流、时间流、后期融合的内部机制 | 用成熟骨干快速搭出可训练的双流基线 |
| 实现方式 | `L1`：从零手写双流 CNN，并完整 mini training | `E2`：基于 `torchvision.models.resnet18` 组装双流模型 |
| 代码量 | 中等，组件拆开讲清楚 | 更少，复用成熟模块 |
| 适合场景 | 面试准备、原理掌握、研究原型 | 工程验证、快速 baseline、结构迁移 |

> 两条路径不是两套无关代码，而是同一个 Two-Stream 思想在不同抽象层级上的表达。

## 1. 论文背景与任务定位

### 1.1 Two-Stream 到底解决什么问题？

视频动作识别既要看“画面里是什么”，也要看“目标是怎么动的”。
原始 Two-Stream 把这两个问题拆开：

- **空间流（Spatial Stream）**：输入单帧 RGB 图像，负责外观与场景线索
- **时间流（Temporal Stream）**：输入多帧光流堆叠，负责运动模式线索
- **后期融合（Late Fusion）**：把两条流的分类分数做加权平均

这让模型既保留了 2D CNN 的成熟训练经验，又显式引入了运动信息。

### 1.2 为什么这个 toy 任务适合演示 Two-Stream？

这里不用 UCF-101 或 HMDB-51，而是构造一个小型合成视频数据集：

- 外观由 **shape** 决定：`square / plus / x`
- 运动由 **direction** 决定：`left / right`
- 类别是两者的组合：共 6 类

这样设计后：

- 单帧更容易看出 **shape**
- 帧差分更容易看出 **direction**
- 融合后最容易恢复完整类别

### 1.3 本 Notebook 的范围

本 Notebook 覆盖：

- 双流架构的核心数据流
- 光流堆叠的可运行近似（用 frame difference 代替 dense optical flow）
- 学习路径的手写实现
- 工程路径的 `torchvision` 版本实现
- 训练 vs 推理差异、结果对比、面试表达

本 Notebook 不覆盖：

- 真实 UCF-101 benchmark 复现
- TV-L1 / RAFT 等真实光流提取工程
- 多 GPU 训练
- 论文里的 25×10 全量测试时增强复现

## 2. 最小必要理论

### 2.1 本 Notebook 只保留最必要的公式

Two-Stream 的核心不是复杂数学，而是**信息分流**：

1. 从视频中取一个代表性的空间视图，送入空间流
2. 从视频中构造时间差分堆叠，送入时间流
3. 将两条流的 logits 做后期融合

我们这里用帧差分近似论文中的光流堆叠：

$$
\tilde{F}_t = I_{t+1} - I_t
$$

后期融合公式：

$$
\hat{y} = \alpha \cdot z_{\text{spatial}} + (1 - \alpha) \cdot z_{\text{temporal}}
$$

其中：

- $I_t$：第 $t$ 帧图像
- $\tilde{F}_t$：第 $t$ 个时间差分图
- $z_{\text{spatial}}$：空间流输出 logits
- $z_{\text{temporal}}$：时间流输出 logits
- $\alpha$：空间流权重

### 2.2 训练 vs 推理为什么不同？

Two-Stream 论文里，训练和推理的数据使用方式不同：

- **训练**：随机 crop、随机采样，提高鲁棒性
- **推理**：密集采样、多视图平均，提高稳定性

本 Notebook 保留这个思想，但做了 CPU / Colab 友好的简化：

- 训练：单随机 crop
- 推理：3 个确定性视图平均

完整理论细节（例如 trajectory stacking、光流压缩、双向光流）请回到配套理论 `.md` 阅读。

## 3. 组件映射表（从论文到代码）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Spatial Stream ConvNet | 手写 `SpatialStreamCNN` | `resnet18` 单帧分支 | Runnable |
| Temporal Stream ConvNet | 手写 `TemporalStreamCNN` | `resnet18` 时间分支 | Runnable |
| Dense optical flow stacking | 用 `compute_frame_diffs()` 做轻量近似 | 同一预处理逻辑复用 | Runnable |
| Naive stacking（固定位置堆叠） | 用固定 crop + 差分堆叠近似 | 无单独高层 API | Runnable |
| Trajectory stacking（沿轨迹堆叠） | Markdown 解释，不在 toy 数据上完整实现 | 无单独高层 API | Explain-only |
| Late Fusion | 显式实现 `alpha * spatial + (1-alpha) * temporal` | 工程路径同式复用 | Runnable |
| 训练时数据增强 | `make_single_view(training=True)` 中的随机 crop | 同一逻辑复用 | Runnable |
| 测试时 dense sampling | `predict_multiview()` 里 3-view averaging | 同一逻辑复用 | Illustrative |
| Optical flow compression | Markdown 解释原论文做法 | 外部预处理流程 | Explain-only |
| Bi-directional flow | Markdown 解释“通道数翻倍”思路 | 需手动扩展时间分支输入通道 | Explain-only |

## 4. 数据准备

真实视频动作识别数据集太大，不适合一份教学 notebook 直接跑通。
这里构造一个 **Two-Stream-friendly synthetic video dataset**：

- 每条视频 9 帧，大小 `64 × 64`
- 共有 6 类：`square_left`、`square_right`、`plus_left`、`plus_right`、`x_left`、`x_right`
- `shape` 提供外观线索，`direction` 提供运动线索
- 所有样本都可直接在 CPU 或免费 Colab 上训练

这个数据集不追求真实世界语义，只追求一件事：**把 Two-Stream 的“外观 + 运动”分工讲清楚。**

In [ ]:
CLASS_SPECS = [
    ("square", "left"),
    ("square", "right"),
    ("plus", "left"),
    ("plus", "right"),
    ("x", "left"),
    ("x", "right"),
]


class SyntheticTwoStreamDataset(Dataset):
    def __init__(
        self,
        n_samples_per_class=30,
        n_frames=9,
        canvas_size=64,
        shape_size=12,
        seed=42,
    ):
        super().__init__()
        self.n_frames = n_frames
        self.canvas_size = canvas_size
        self.shape_size = shape_size
        self.class_specs = CLASS_SPECS
        self.class_names = [f"{shape}_{direction}" for shape, direction in CLASS_SPECS]

        template_names = {shape for shape, _ in CLASS_SPECS}
        self.templates = {
            name: self._build_template(name, shape_size) for name in template_names
        }

        rng = np.random.RandomState(seed)
        videos, labels = [], []
        for label, (shape_name, direction) in enumerate(self.class_specs):
            for _ in range(n_samples_per_class):
                videos.append(self._make_video(shape_name, direction, rng))
                labels.append(label)

        self.videos = torch.stack(videos).float()  # (N, T, H, W)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def _build_template(self, name, size):
        canvas = np.zeros((size, size), dtype=np.float32)

        if name == "square":
            canvas[1:-1, 1:-1] = 0.20
            canvas[1:3, :] = 1.0
            canvas[-3:-1, :] = 1.0
            canvas[:, 1:3] = 1.0
            canvas[:, -3:-1] = 1.0
        elif name == "plus":
            center = size // 2
            canvas[center - 1 : center + 1, :] = 1.0
            canvas[:, center - 1 : center + 1] = 1.0
            canvas[2:-2, 2:-2] = np.maximum(canvas[2:-2, 2:-2], 0.25)
        elif name == "x":
            for i in range(size):
                for offset in (-1, 0, 1):
                    j1 = i + offset
                    j2 = size - 1 - i + offset
                    if 0 <= j1 < size:
                        canvas[i, j1] = 1.0
                    if 0 <= j2 < size:
                        canvas[i, j2] = 1.0
            center = size // 2
            canvas[center - 1 : center + 1, center - 1 : center + 1] = 0.35
        else:
            raise ValueError(f"Unsupported template name: {name}")

        return np.clip(canvas, 0.0, 1.0)

    def _make_video(self, shape_name, direction, rng):
        dx = -3 if direction == "left" else 3
        mid = self.n_frames // 2
        max_pos = self.canvas_size - self.shape_size

        shifts = [dx * (t - mid) for t in range(self.n_frames)]
        x_mid_min = -min(shifts)
        x_mid_max = max_pos - max(shifts)

        x_mid = rng.randint(x_mid_min, x_mid_max + 1)
        y = rng.randint(6, max_pos - 5)
        x0 = x_mid - dx * mid

        amplitude = float(rng.uniform(0.90, 1.10))
        template = np.clip(self.templates[shape_name] * amplitude, 0.0, 1.0)

        frames = []
        for t in range(self.n_frames):
            frame = np.zeros((self.canvas_size, self.canvas_size), dtype=np.float32)
            x = x0 + dx * t
            frame[y : y + self.shape_size, x : x + self.shape_size] = template
            frame += rng.normal(0.0, 0.02, size=frame.shape).astype(np.float32)
            frame = np.clip(frame, 0.0, 1.0)
            frames.append(torch.from_numpy(frame))

        return torch.stack(frames, dim=0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], int(self.labels[idx].item())


dataset = SyntheticTwoStreamDataset(n_samples_per_class=30, n_frames=9, canvas_size=64, shape_size=12, seed=42)
print(f"Dataset size: {len(dataset)}")
print(f"Video tensor shape: {tuple(dataset.videos.shape)}")
print(f"Class names: {dataset.class_names}")

In [ ]:
def compute_frame_diffs(video_tensor):
    if video_tensor.dim() == 3:
        return video_tensor[1:] - video_tensor[:-1]
    if video_tensor.dim() == 4:
        return video_tensor[:, 1:] - video_tensor[:, :-1]
    raise ValueError(f"Unsupported tensor shape: {tuple(video_tensor.shape)}")


CROP_SIZE = 56
TRAIN_VIEWS = 1
EVAL_VIEWS = 3


def get_eval_crop_positions(size, crop_size=CROP_SIZE):
    margin = size - crop_size
    return [(0, 0), (margin // 2, margin // 2), (margin, margin)]


def make_single_view(video, training=True, view_index=0, crop_size=CROP_SIZE, return_meta=False):
    num_frames, height, width = video.shape
    max_top = height - crop_size
    max_left = width - crop_size

    if training:
        top = random.randint(0, max_top)
        left = random.randint(0, max_left)
    else:
        top, left = get_eval_crop_positions(height, crop_size)[view_index]

    cropped = video[:, top : top + crop_size, left : left + crop_size]  # (T, H, W)
    spatial = cropped[num_frames // 2].unsqueeze(0)                     # (1, H, W)
    temporal = compute_frame_diffs(cropped)                             # (T-1, H, W)

    if return_meta:
        meta = {"top": top, "left": left, "view_index": view_index}
        return spatial, temporal, meta
    return spatial, temporal


def make_batch_views(videos, training=True, num_views=1):
    spatial_views, temporal_views = [], []
    for video in videos:
        for view_index in range(num_views):
            spatial, temporal = make_single_view(video, training=training, view_index=view_index)
            spatial_views.append(spatial)
            temporal_views.append(temporal)

    spatial_tensor = torch.stack(spatial_views).float()   # (B*num_views, 1, H, W)
    temporal_tensor = torch.stack(temporal_views).float() # (B*num_views, T-1, H, W)
    return spatial_tensor, temporal_tensor


split_generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=split_generator)

train_loader = DataLoader(
    train_dataset,
    batch_size=24,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=48,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

sample_videos, sample_labels = next(iter(train_loader))
spatial_batch, temporal_batch = make_batch_views(sample_videos, training=True, num_views=1)

print(f"Train batch videos: {tuple(sample_videos.shape)}")
print(f"Spatial batch: {tuple(spatial_batch.shape)}")
print(f"Temporal batch: {tuple(temporal_batch.shape)}")
print(f"Temporal channels: {temporal_batch.shape[1]}")

In [ ]:
def show_video_and_diffs(video, label_name, n_show=4):
    frame_ids = np.linspace(0, video.shape[0] - 1, n_show, dtype=int)
    diff_tensor = compute_frame_diffs(video)
    diff_ids = np.linspace(0, diff_tensor.shape[0] - 1, n_show, dtype=int)

    fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 5))
    for col, frame_id in enumerate(frame_ids):
        axes[0, col].imshow(video[frame_id], cmap="gray", vmin=0.0, vmax=1.0)
        axes[0, col].set_title(f"Frame {frame_id}")
        axes[0, col].axis("off")

    for col, diff_id in enumerate(diff_ids):
        axes[1, col].imshow(diff_tensor[diff_id], cmap="RdBu_r", vmin=-1.0, vmax=1.0)
        axes[1, col].set_title(f"Diff {diff_id}->{diff_id + 1}")
        axes[1, col].axis("off")

    fig.suptitle(f"Class: {label_name}", y=1.02)
    plt.tight_layout()
    plt.show()


sample_video, sample_label = dataset[0]
show_video_and_diffs(sample_video, dataset.class_names[sample_label], n_show=4)

## 5. Shared Components

下面的超参数、训练循环和评估逻辑会被两条路径共同复用。

这样做有两个目的：

1. 保证比较公平——同一份数据、同一套训练与验证协议
2. 让“学习路径 vs 工程路径”的差别只集中在**模型抽象层级**上

In [ ]:
NUM_CLASSES = len(dataset.class_names)
NUM_FRAMES = dataset.n_frames
TEMPORAL_CHANNELS = NUM_FRAMES - 1

# 为了让学习路径和工程路径可直接比较，训练预算保持一致
COMMON_EPOCHS = 3
COMMON_LR = 1e-3
LEARNING_EPOCHS = COMMON_EPOCHS
ENGINEERING_EPOCHS = COMMON_EPOCHS
LEARNING_LR = COMMON_LR
ENGINEERING_LR = COMMON_LR
FUSION_ALPHA = 0.5

print(
    f"NUM_CLASSES={NUM_CLASSES}, NUM_FRAMES={NUM_FRAMES}, "
    f"TEMPORAL_CHANNELS={TEMPORAL_CHANNELS}, COMMON_EPOCHS={COMMON_EPOCHS}, COMMON_LR={COMMON_LR}"
)

In [ ]:
def train_one_epoch(model, loader, optimizer):
    criterion = nn.CrossEntropyLoss()
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0

    for videos, labels in loader:
        labels = labels.to(device)
        spatial_inputs, temporal_inputs = make_batch_views(videos, training=True, num_views=TRAIN_VIEWS)
        spatial_inputs = spatial_inputs.to(device)
        temporal_inputs = temporal_inputs.to(device)

        optimizer.zero_grad()
        logits = model(spatial_inputs, temporal_inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return total_loss / total_count, total_correct / total_count


@torch.no_grad()
def evaluate(model, loader, num_views=EVAL_VIEWS):
    model.eval()
    total_correct, total_count = 0, 0

    for videos, labels in loader:
        labels = labels.to(device)
        batch_size = labels.size(0)

        spatial_inputs, temporal_inputs = make_batch_views(videos, training=False, num_views=num_views)
        spatial_inputs = spatial_inputs.to(device)
        temporal_inputs = temporal_inputs.to(device)

        logits = model(spatial_inputs, temporal_inputs)
        logits = logits.view(batch_size, num_views, -1).mean(dim=1)

        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return total_correct / total_count


def fit_model(model, train_loader, val_loader, epochs, lr, tag):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_acc = evaluate(model, val_loader, num_views=EVAL_VIEWS)
        elapsed = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"[{tag}] epoch {epoch:02d}/{epochs} | "
            f"loss={train_loss:.4f} | train_acc={train_acc:.2%} | "
            f"val_acc={val_acc:.2%} | time={elapsed:.1f}s"
        )

    return history


@torch.no_grad()
def predict_multiview(model, video, num_views=EVAL_VIEWS):
    model.eval()
    spatial_inputs, temporal_inputs = make_batch_views(video.unsqueeze(0), training=False, num_views=num_views)
    spatial_inputs = spatial_inputs.to(device)
    temporal_inputs = temporal_inputs.to(device)

    logits = model(spatial_inputs, temporal_inputs)
    logits = logits.view(1, num_views, -1).mean(dim=1)
    probs = logits.softmax(dim=1)
    return logits.squeeze(0).cpu(), probs.squeeze(0).cpu()

## 6. 学习路径：L1 Full Mini Training

学习路径选择 **L1**，原因很直接：

- Two-Stream 的核心结构并不依赖超大模型
- toy 数据集足以支持从零训练一个小型双流 CNN
- 在免费 Colab / CPU 环境里也能跑通完整训练闭环

下面按数据流顺序实现：

1. 空间流 CNN
2. 时间流 CNN
3. 后期融合
4. 训练与验证

### 6.1 空间流（Spatial Stream）

空间流只看一个代表帧，负责建模**外观信息**。

它学习的是：

$$
 z_{\text{spatial}} = f_{\text{spatial}}(x_{\text{mid}})
$$

其中：

- 输入 $x_{\text{mid}} \in \mathbb{R}^{1 \times H \times W}$
- 输出 $z_{\text{spatial}} \in \mathbb{R}^{C}$

直觉上，它更容易区分 `square / plus / x` 这样的外观差异。
对于 `left / right` 这种只靠运动才能稳定判断的属性，空间流天然更吃亏。

In [ ]:
class SpatialStreamCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),   # (B, 1, 56, 56) -> (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # (B, 32, 28, 28) -> (B, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # (B, 64, 14, 14) -> (B, 128, 7, 7)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        features = self.features(x)
        return self.classifier(features)


spatial_demo_model = SpatialStreamCNN(NUM_CLASSES)
print(f"Spatial stream parameters: {sum(p.numel() for p in spatial_demo_model.parameters()):,}")

### 6.2 时间流（Temporal Stream）

时间流把连续帧的运动变化堆叠为多通道输入，负责建模**运动信息**。

在原论文里，输入是密集光流堆叠；这里为了可运行性，用帧差分近似：

$$
\tilde{F}_t = I_{t+1} - I_t
$$

再把所有时间差分堆叠成：

$$
X_{\text{temporal}} \in \mathbb{R}^{(T-1) \times H \times W}
$$

直觉上，它更容易区分 `left / right`，因为方向信息直接体现在差分图里。

In [ ]:
class TemporalStreamCNN(nn.Module):
    def __init__(self, temporal_channels, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(temporal_channels, 32, kernel_size=3, stride=2, padding=1),   # (B, 8, 56, 56) -> (B, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),                  # (B, 32, 28, 28) -> (B, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),                 # (B, 64, 14, 14) -> (B, 128, 7, 7)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        features = self.features(x)
        return self.classifier(features)


temporal_demo_model = TemporalStreamCNN(TEMPORAL_CHANNELS, NUM_CLASSES)
print(f"Temporal stream parameters: {sum(p.numel() for p in temporal_demo_model.parameters()):,}")

### 6.3 后期融合（Late Fusion）

Two-Stream 最经典的地方就在于：两条流先独立输出 logits，再做加权融合。

$$
\hat{y} = \alpha \cdot z_{\text{spatial}} + (1 - \alpha) \cdot z_{\text{temporal}}
$$

为什么这里融合的是 **logits** 而不是 softmax 概率？

- logits 更接近“未归一化证据”
- 在 logits 层加权更稳定，信息损失更少
- 这也更接近原论文的“score fusion”表达

In [ ]:
class TwoStreamLearningModel(nn.Module):
    def __init__(self, num_classes, temporal_channels, alpha=0.5):
        super().__init__()
        self.spatial = SpatialStreamCNN(num_classes)
        self.temporal = TemporalStreamCNN(temporal_channels, num_classes)
        self.alpha = alpha

    def forward(self, spatial_input, temporal_input, return_parts=False):
        spatial_logits = self.spatial(spatial_input)      # (B, C)
        temporal_logits = self.temporal(temporal_input)   # (B, C)
        fused_logits = self.alpha * spatial_logits + (1.0 - self.alpha) * temporal_logits

        if return_parts:
            return fused_logits, spatial_logits, temporal_logits
        return fused_logits


learning_model = TwoStreamLearningModel(
    num_classes=NUM_CLASSES,
    temporal_channels=TEMPORAL_CHANNELS,
    alpha=FUSION_ALPHA,
).to(device)

fused_demo, spatial_demo, temporal_demo = learning_model(
    spatial_batch[:4].to(device),
    temporal_batch[:4].to(device),
    return_parts=True,
)

print(f"Spatial logits shape: {tuple(spatial_demo.shape)}")
print(f"Temporal logits shape: {tuple(temporal_demo.shape)}")
print(f"Fused logits shape: {tuple(fused_demo.shape)}")

## 7. 训练 vs 推理的区别

原论文里，训练和推理并不是“完全同一条路径”。
差异主要在**如何从同一条视频取输入视图**。

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | 单随机 crop，再取中间帧 + 帧差分 | 同样单随机 crop，只是 backbone 更深 |
| 推理 | 3 个确定性视图平均 logits | 同样 3-view averaging |
| 目的 | 提高鲁棒性、缓解过拟合 | 提高输出稳定性与可重复性 |

这说明：**同一个算法思想，在不同抽象层级上都保留了训练 / 推理分离。**

In [ ]:
video_example, label_example = dataset[0]
_, _, train_meta = make_single_view(video_example, training=True, return_meta=True)
_, _, eval_meta_0 = make_single_view(video_example, training=False, view_index=0, return_meta=True)
_, _, eval_meta_1 = make_single_view(video_example, training=False, view_index=1, return_meta=True)
_, _, eval_meta_2 = make_single_view(video_example, training=False, view_index=2, return_meta=True)

print(f"Class example: {dataset.class_names[label_example]}")
print(f"Train crop meta: {train_meta}")
print(f"Eval crop meta #0: {eval_meta_0}")
print(f"Eval crop meta #1: {eval_meta_1}")
print(f"Eval crop meta #2: {eval_meta_2}")

## 8. 学习路径训练

现在正式训练手写版 Two-Stream。

由于类别同时依赖 **shape** 与 **direction**，这个 toy 任务天然适合检验：

- 空间流是否学到外观
- 时间流是否学到运动
- 融合后是否比单一路径更完整

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

learning_model = TwoStreamLearningModel(
    num_classes=NUM_CLASSES,
    temporal_channels=TEMPORAL_CHANNELS,
    alpha=FUSION_ALPHA,
).to(device)

learning_history = fit_model(
    learning_model,
    train_loader,
    val_loader,
    epochs=LEARNING_EPOCHS,
    lr=LEARNING_LR,
    tag="Learning",
)
learning_val_acc = evaluate(learning_model, val_loader, num_views=EVAL_VIEWS)
print(f"Learning path final val acc: {learning_val_acc:.2%}")

## 9. 工程路径：E2 Limited Tooling + Train

Two-Stream 并没有一个像 `transformers.pipeline()` 那样的成熟统一高层 API。
因此这里选择 **E2：有限工具链**：

- 复用 `torchvision.models.resnet18`
- 我们自己保留双流结构与 late fusion
- 继续沿用学习路径里的同一份数据与训练 / 推理协议
- **并且使用相同的训练轮数与学习率**，保证对比尽量公平

### 官方 API 选择说明

根据当前 `torchvision` 官方文档：

- `resnet18(weights=ResNet18_Weights.DEFAULT)`：加载默认 ImageNet 权重
- `resnet18(weights=None)`：随机初始化

为了保证 notebook 在**离线环境也能直接跑通**，这里默认 `use_pretrained=False`。
同时，为了兼容较旧的 `torchvision` 版本，代码里对 `ResNet18_Weights` 做了 fallback 处理。

### 学习路径实现 vs 工程路径对应

| 学习路径（手写） | 工程路径（库 / 封装） | 说明 |
|---|---|---|
| `SpatialStreamCNN` | `resnet18` 单帧分支 | 由轻量 CNN 换成成熟图像骨干 |
| `TemporalStreamCNN` | 修改首层卷积后的 `resnet18` 时间分支 | 多通道时间输入仍然保留 |
| `TwoStreamLearningModel` | `TwoStreamResNet` | 后期融合逻辑保持不变 |
| 手写卷积细节 | `torchvision` 已实现残差块和分类头 | 工程上更省代码、更稳 |

In [ ]:
def build_resnet18_stream(in_channels, num_classes, use_pretrained=False):
    if ResNet18_Weights is not None:
        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        backbone = resnet18(weights=weights)
    else:
        backbone = resnet18(pretrained=use_pretrained)

    if in_channels != 3:
        old_conv = backbone.conv1
        new_conv = nn.Conv2d(
            in_channels,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        if use_pretrained:
            mean_weight = old_conv.weight.data.mean(dim=1, keepdim=True)
            new_conv.weight.data.copy_(mean_weight.repeat(1, in_channels, 1, 1))

        backbone.conv1 = new_conv

    backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
    return backbone


class TwoStreamResNet(nn.Module):
    def __init__(self, num_classes, temporal_channels, alpha=0.5, use_pretrained=False):
        super().__init__()
        self.spatial = build_resnet18_stream(1, num_classes, use_pretrained=use_pretrained)
        self.temporal = build_resnet18_stream(temporal_channels, num_classes, use_pretrained=use_pretrained)
        self.alpha = alpha

    def forward(self, spatial_input, temporal_input, return_parts=False):
        spatial_logits = self.spatial(spatial_input)
        temporal_logits = self.temporal(temporal_input)
        fused_logits = self.alpha * spatial_logits + (1.0 - self.alpha) * temporal_logits

        if return_parts:
            return fused_logits, spatial_logits, temporal_logits
        return fused_logits


engineering_model = TwoStreamResNet(
    num_classes=NUM_CLASSES,
    temporal_channels=TEMPORAL_CHANNELS,
    alpha=FUSION_ALPHA,
    use_pretrained=False,
).to(device)

print(f"Engineering model parameters: {sum(p.numel() for p in engineering_model.parameters()):,}")

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

engineering_model = TwoStreamResNet(
    num_classes=NUM_CLASSES,
    temporal_channels=TEMPORAL_CHANNELS,
    alpha=FUSION_ALPHA,
    use_pretrained=False,
).to(device)

engineering_history = fit_model(
    engineering_model,
    train_loader,
    val_loader,
    epochs=ENGINEERING_EPOCHS,
    lr=ENGINEERING_LR,
    tag="Engineering",
)
engineering_val_acc = evaluate(engineering_model, val_loader, num_views=EVAL_VIEWS)
print(f"Engineering path final val acc: {engineering_val_acc:.2%}")

## 10. 学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能直接看到空间流、时间流、后期融合分别在做什么 | 能理解如何把论文结构迁移到成熟 backbone |
| 代码量 | 中等，组件拆得更细 | 更少，核心逻辑更集中 |
| 灵活性 | 高，任何层和融合方式都能自由改 | 中，受 `resnet18` 结构约束 |
| 稳定性 | 中，完全依赖手写实现细节 | 高，骨干网络实现更成熟 |
| 工业适配度 | 更适合教学、研究原型、面试展示 | 更适合 baseline、快速验证、结构迁移 |
| 适用场景 | 面试准备、论文复现入门、理解机制 | 工程试验、迁移到真实数据、继续换骨干 |

### 两条路径的边界

**学习路径的收益：**

- 看懂 Two-Stream 为什么要拆成两条流
- 能清楚解释“单帧负责外观、差分负责运动”
- 面试时可以逐组件讲清楚数据流与张量形状

**工程路径的收益：**

- 快速搭出更强的 baseline
- 更接近真实项目中的“复用成熟骨干 + 改任务头”思路
- 后续换成更强 backbone 更自然

In [ ]:
def plot_histories(learning_history, engineering_history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(learning_history["train_loss"], marker="o", label="Learning")
    axes[0].plot(engineering_history["train_loss"], marker="s", label="Engineering")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(learning_history["val_acc"], marker="o", label="Learning")
    axes[1].plot(engineering_history["val_acc"], marker="s", label="Engineering")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


@torch.no_grad()
def get_branch_predictions(model, video, num_views=EVAL_VIEWS):
    model.eval()
    spatial_inputs, temporal_inputs = make_batch_views(video.unsqueeze(0), training=False, num_views=num_views)
    spatial_inputs = spatial_inputs.to(device)
    temporal_inputs = temporal_inputs.to(device)

    fused_logits, spatial_logits, temporal_logits = model(
        spatial_inputs,
        temporal_inputs,
        return_parts=True,
    )

    fused_logits = fused_logits.view(1, num_views, -1).mean(dim=1).squeeze(0).cpu()
    spatial_logits = spatial_logits.view(1, num_views, -1).mean(dim=1).squeeze(0).cpu()
    temporal_logits = temporal_logits.view(1, num_views, -1).mean(dim=1).squeeze(0).cpu()

    return (
        spatial_logits.argmax().item(),
        temporal_logits.argmax().item(),
        fused_logits.argmax().item(),
    )


plot_histories(learning_history, engineering_history)
print(f"Learning validation accuracy   : {learning_val_acc:.2%}")
print(f"Engineering validation accuracy: {engineering_val_acc:.2%}")

print("\nPrediction examples:")
for sample_index in range(3):
    video, label = val_dataset[sample_index]
    learning_pred = get_branch_predictions(learning_model, video)
    engineering_pred = get_branch_predictions(engineering_model, video)

    print(
        f"sample {sample_index:02d} | gt={dataset.class_names[label]:<12} | "
        f"learning fused={dataset.class_names[learning_pred[2]]:<12} | "
        f"engineering fused={dataset.class_names[engineering_pred[2]]:<12}"
    )

video_vis, label_vis = val_dataset[0]
show_video_and_diffs(video_vis, dataset.class_names[label_vis], n_show=4)

## 11. 面试与项目表达

### 高频面试题

**Q1：为什么 Two-Stream 要拆成空间流和时间流？**

- 因为视频里的“外观”和“运动”本来就是两种不同信息
- 单帧图像更适合提取物体、背景、姿态等静态线索
- 光流或差分更适合提取方向、速度、动作模式等动态线索
- 拆开之后可以直接复用成熟 2D CNN，再做 score fusion

**Q2：为什么时间流常常比空间流更重要？**

- 很多动作类别真正决定类别的是“怎么动”，不是“长什么样”
- 挥拍、投掷、挥手这类动作，单帧画面可能非常像，但运动模式差别很大
- 这也是原论文里 temporal stream 表现很强的原因

**Q3：Late Fusion 的优点和局限是什么？**

- 优点：实现简单、两条流可独立训练、工程门槛低
- 局限：两条流直到 logits 层才交流，学不到更细粒度的时空交互
- 所以后续工作会继续探索 early fusion、slow fusion、3D 时空建模

**Q4：为什么推理要做 dense sampling 或 multi-view averaging？**

- 单个 crop、单个时间点都可能带来很大偶然性
- 多位置、多时间视图平均，相当于让推理阶段更稳
- 它本质上是在降低采样噪声，提高可重复性

**Q5：帧差分和真实光流有什么差别？**

- 帧差分只看像素值变化，计算极快
- 真实光流估计每个像素的位移向量，更接近真正的运动场
- 帧差分适合教学演示；真实动作识别通常仍会使用更专业的光流算法

**Q6：Two-Stream 的主要瓶颈是什么？**

- 最大工程瓶颈是光流预计算成本高
- 算得慢、存得大、难以端到端优化
- 这也是为什么后续很多方法转向 TSN、I3D、SlowFast、Video Transformer

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | Two-Stream 的核心思想是什么？ | 把外观和运动拆成两条流，最后做 score fusion |
| 2 | 空间流看什么？ | 看单帧外观与场景线索 |
| 3 | 时间流看什么？ | 看多帧运动模式，通常由光流或差分提供 |
| 4 | 为什么要 late fusion？ | 简单、稳定、易训练，但交互发生得太晚 |
| 5 | 为什么推理要多视图平均？ | 降低单次采样偶然性，提高预测稳定性 |
| 6 | Two-Stream 的短板是什么？ | 光流预计算代价高，时空交互不够早 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样回答：

- **学习深度**：我从零实现了 Two-Stream 的空间流、时间流和后期融合，能解释每条流分别建模什么信息
- **工程能力**：我用 `torchvision` 的 `resnet18` 组装了一个可训练的双流 baseline，理解了如何把论文结构迁移到成熟骨干上
- **对比思考**：我能说清学习路径和工程路径的差别，也能解释为什么 Two-Stream 后来会演化到 TSN、I3D、SlowFast 这类方法

### 延伸阅读与对比

| 模型 | 核心思想 | 优势 | 局限 |
|---|---|---|---|
| Two-Stream | RGB 流 + 光流流 + late fusion | 结构清晰、效果经典 | 光流成本高 |
| TSN | 稀疏时间采样 + segment consensus | 更轻量、覆盖长时序 | 局部时序建模较弱 |
| I3D | 3D 卷积直接建模时空 | 时空建模更强 | 算力更高 |
| SlowFast | 快慢双路径处理不同时间频率 | 时空分工更自然 | 结构更复杂 |

### 进阶探索方向

- 把当前 frame difference 替换成真实 optical flow
- 把 late fusion 改成 feature-level fusion
- 把 2D backbone 升级成更强的视频骨干
- 在真实小数据集上做更接近论文设定的实验

### 参考来源

- 原论文 PDF：<https://www.robots.ox.ac.uk/~vgg/publications/2014/Simonyan14b/simonyan14b.pdf>
- Oxford 项目页：<https://www.robots.ox.ac.uk/~vgg/publications/2014/Simonyan14b/>
- NeurIPS 页面：<https://proceedings.neurips.cc/paper/5353-two-stream-convolutional>
- 后续融合论文（CVPR 2016）：<https://www.cv-foundation.org/openaccess/content_cvpr_2016/papers/Feichtenhofer_Convolutional_Two-Stream_Network_CVPR_2016_paper.pdf>